In [1]:
#Imports
import pandas as pd
import numpy as np
import os
import pickle
from glob import glob
from tqdm import tqdm

import sys
sys.path.insert(0, "/group/glastonbury/soumick/MyCodes/GitLab/tricorder")

class MLAnalyses:
    def __init__(self) -> None:
        pass

In [2]:
assoc = pd.read_csv("/group/glastonbury/Rupali/prova_assoc.csv")
modres = pd.read_csv("/group/glastonbury/Rupali/prova_modelresults.csv")

In [6]:
assoc

,Pheno,Association,Latent,EffectSize,StdError,P,ExpID,ExpParams,ResType
0,Left_kidney_volume,Best,Z19,0.073303,0.003136,8.709502e-120,BaseAE,VIF_TScale_FScale,Abdominal_organ_composition_82779_MD_01_08_202...
1,Left_kidney_volume,LowestP,Z19,0.073303,0.003136,8.709502e-120,BaseAE,VIF_TScale_FScale,Abdominal_organ_composition_82779_MD_01_08_202...
2,Liver_PDFF_(fat_fraction)_-_gradient_echo,Best,Z19,0.077944,0.008583,1.706042e-19,BaseAE,VIF_TScale_FScale,Abdominal_organ_composition_82779_MD_01_08_202...
3,Liver_PDFF_(fat_fraction)_-_gradient_echo,LowestP,Z19,0.077944,0.008583,1.706042e-19,BaseAE,VIF_TScale_FScale,Abdominal_organ_composition_82779_MD_01_08_202...
4,Liver_PDFF_(fat_fraction),Best,Z19,0.091460,0.003180,1.867800e-179,BaseAE,VIF_TScale_FScale,Abdominal_organ_composition_82779_MD_01_08_202...
...,...,...,...,...,...,...,...,...,...
1991,Total_trunk_fat_volume,LowestP,Z34_mag,0.238435,0.012602,2.209299e-79,CAE_DualCoordsL1_polar,VIF_TScale_FScale,Abdominal_composition_82779_MD_01_08_2024_16_5...
1992,Visceral_adipose_tissue_volume_(VAT),Best,Z102_mag,0.169958,0.007481,1.192722e-113,CAE_DualCoordsL1_polar,VIF_TScale_FScale,Abdominal_composition_82779_MD_01_08_2024_16_5...
1993,Visceral_adipose_tissue_volume_(VAT),LowestP,Z74_mag,-0.172033,0.007479,1.886834e-116,CAE_DualCoordsL1_polar,VIF_TScale_FScale,Abdominal_composition_82779_MD_01_08_2024_16_5...
1994,Weight-to-muscle_ratio,Best,Z62_mag,-0.029849,0.009370,1.446061e-03,CAE_DualCoordsL1_polar,VIF_TScale_FScale,Abdominal_composition_82779_MD_01_08_2024_16_5...


In [2]:
#Evaluate associations
def standardise(value, mean, std):
    return (value - mean) / std

def get_best_assoc(models_dict):
    p_values = [model_info['p-value'] for model_info in models_dict.values()]
    std_errors = [model_info['StdError'] for model_info in models_dict.values()]
    effect_sizes = [model_info['EffectSize'] for model_info in models_dict.values()]
    
    mean_p_value = np.mean(p_values)
    std_p_value = np.std(p_values)
    
    mean_std_error = np.mean(std_errors)
    std_std_error = np.std(std_errors)
    
    mean_effect_size = np.mean(effect_sizes)
    std_effect_size = np.std(effect_sizes)
    
    best_model = None
    best_model_score = float('inf')  # Initialize with infinity to find the minimum score

    for model_name, model_info in models_dict.items():
        effect_size = model_info['EffectSize']
        std_error = model_info['StdError']
        p_value = model_info['p-value']
        
        # Standardise each metric
        standardised_p_value = standardise(p_value, mean_p_value, std_p_value)
        standardised_std_error = standardise(std_error, mean_std_error, std_std_error)
        standardised_effect_size = standardise(effect_size, mean_effect_size, std_effect_size)
        
        # Combine the standardised metrics into a composite score
        # You can adjust the weights if you want to prioritize one metric over others
        model_score = standardised_p_value + standardised_std_error - standardised_effect_size
        
        # Compare and update the best model if the current one has a better score
        if model_score < best_model_score:
            best_model_score = model_score
            best_model = model_name

    return best_model, models_dict[best_model]

def get_lowestP_assoc(models_dict):
    best_model = None
    lowest_p_value = float('inf')  

    for model_name, model_info in models_dict.items():
        p_value = model_info['p-value']
        
        if p_value < lowest_p_value:
            lowest_p_value = p_value
            best_model = model_name

    return best_model, models_dict[best_model]

def get_all_assoc(models_dict, pheno):
    res_dict = []
    for model_name, model_info in models_dict.items():
        res_dict.append({
            'Pheno': pheno,
            "EffectSize": model_info['EffectSize'],
            "StdError": model_info['StdError'],
            "P": model_info['p-value'],
            "Latent": model_name,
            "TrainSize": model_info['TrainSize']
        })
    return res_dict

def get_assoc_consolidated(best_assoc_latent, best_assoc_model, lowestP_assoc_latent, lowestP_assoc_model, pheno):
    res_dict = []

    #best_asso
    res_dict.append({'Pheno': pheno, 'Association': 'Best', 
                     'Latent': best_assoc_latent, 
                     'EffectSize': best_assoc_model['EffectSize'],
                     'StdError': best_assoc_model['StdError'],
                     'P': best_assoc_model['p-value']})
    
    #lowestP_asso
    res_dict.append({'Pheno': pheno, 'Association': 'LowestP', 
                     'Latent': lowestP_assoc_latent, 
                     'EffectSize': lowestP_assoc_model['EffectSize'],
                     'StdError': lowestP_assoc_model['StdError'],
                     'P': lowestP_assoc_model['p-value']})
    
    return res_dict

In [3]:
#Summerise model results

def consolidate_metrics(folds_dict):
    rows = []

    for fold, metrics in folds_dict.items():
        mse_test = metrics['MSE_TestSet']
        r_squared_test = metrics['R-squared_TestSet']
        row = {
            'Fold': fold,
            'MSE_TestSet': mse_test,
            'R-squared_TestSet': r_squared_test
        }
        rows.append(row)

    return pd.DataFrame(rows)

def get_highest_median_r_consolidate_metrics(res, keys_list):
    best_median_r_squared = -float('inf')  
    for key in keys_list:
        if key in res:
            folds_dict = res[key]
            consolidated_df = consolidate_metrics(folds_dict)
            median_r_squared = consolidated_df['R-squared_TestSet'].median()
            if median_r_squared > best_median_r_squared:
                best_consolidated_df = consolidated_df
                best_median_r_squared = median_r_squared
                best_consolidated_df['nFeat'] = int(key.split('_')[-1].replace('nfeat', ''))
    return best_consolidated_df

In [4]:
# def analyse_associations(associations_df, baseline_expid="BaseAE"):
#     sorted_df = associations_df.sort_values(by=['Pheno', 'P'])

#     baseline_wins = set()
#     proposed_wins = set()
#     expid_wins = {}

#     grouped = sorted_df.groupby('Pheno')

#     for pheno, group in grouped:
#         top_row = group.iloc[0]

#         if top_row['ExpID'] == "BaseAE":
#             baseline_wins.add(pheno)
#         else:
#             proposed_wins.add(pheno)

#         if top_row['ExpID'] in expid_wins:
#             expid_wins[top_row['ExpID']] += 1
#         else:
#             expid_wins[top_row['ExpID']] = 1
#     expid_wins_df = pd.DataFrame(list(expid_wins.items()), columns=['ExpID', 'WinCount'])

#     return list(baseline_wins), list(proposed_wins), expid_wins_df

def compare_model_results_with_baseline(res_df, mode="r2", baseline_expid="BaseAE"):
    if mode == "assoc":
        sorted_df = res_df.sort_values(by=['Pheno', 'P'])
        grouped = sorted_df.groupby('Pheno')
    else:
        sorted_df = res_df.sort_values(by=['Pheno', 'Model', 'R-squared_TestSet'], ascending=[True, True, False])
        grouped = sorted_df.groupby(['Pheno', 'Model'])

    baseline_wins = set()
    proposed_wins = set()
    expid_wins = {}

    for identifier, group in grouped:
        top_row = group.iloc[0]

        if top_row['ExpID'] == baseline_expid:
            baseline_wins.add(identifier)
        else:
            proposed_wins.add(identifier)

        if mode == "assoc":
            expid_model_key = top_row['ExpID']
        else:
            expid_model_key = f"{top_row['ExpID']}-{identifier[1]}"

        if expid_model_key in expid_wins:
            expid_wins[expid_model_key] += 1
        else:
            expid_wins[expid_model_key] = 1

    expid_wins_df = pd.DataFrame(list(expid_wins.items()), columns=['ExpID', 'WinCount'])
    return list(baseline_wins), list(proposed_wins), expid_wins_df

In [5]:
exp_root = "/project/ukbblatent/soumick/ML_analyses/F20259v3/v1.1.0_seventh_basket/CAE_Rupali"
out_root = "/group/glastonbury/Rupali/Res4Thesis"

In [6]:
#exp_param = "CVevalV0_TScale" #"VIF_TScale_FScale"
exp_param = "TScale" #"VIF_TScale_FScale"

In [7]:
exps = glob(exp_root + f"/**/{exp_param}*.pkl", recursive=True)
exp = exps[0]

In [32]:
# exps.append("/project/ukbblatent/soumick/ML_analyses/F20259v3/v1.1.0_seventh_basket/CAE_Rupali/BaseAE/VIF_TScale_FScale_BaseAE_Abdominal_composition_82779_MD_01_08_2024_16_54_05_results.pkl")
# exps.append("/project/ukbblatent/soumick/ML_analyses/F20259v3/v1.1.0_seventh_basket/CAE_Rupali/BaseAE/VIF_TScale_FScale_BaseAE_Abdominal_organ_composition_82779_MD_01_08_2024_16_56_07_results.pkl")

In [9]:
def consolidate_res(res, expID, exp_prams, resType):
    all_associations = []
    associations = []
    model_results = []

    mod_keys = sorted(list(res.keys()))
    i = 0
    while i < len(mod_keys):

        #ends with "_RFELinRegrs_CV_nfeat<N>" (or "_RFELinRegrs_CV_nfeat<N>_FeatureSelection")
        if "_RFELinRegrs_CV_nfeat" in mod_keys[i]:
            search_key = mod_keys[i].split("_RFELinRegrs_CV_nfeat")[0]
            lst_models = [mod_keys[i]]
            while (i+1 < len(mod_keys)) and ("_RFELinRegrs_CV_nfeat" in mod_keys[i+1]) and (search_key in mod_keys[i+1]):
                lst_models.append(mod_keys[i+1])
                i += 1
            best_nfeat = get_highest_median_r_consolidate_metrics(res, lst_models)
            best_nfeat['Pheno'] = search_key
            best_nfeat['Model'] = "RFELinRegrs_CV"
            model_results.append(best_nfeat)
            i += 1

        #endswith "_Assoc_dataset"
        elif mod_keys[i].endswith("_Assoc_dataset"):
            best_assoc_latent, best_assoc_model = get_best_assoc(res[mod_keys[i]])
            lowestP_assoc_latent, lowestP_assoc_model = get_lowestP_assoc(res[mod_keys[i]])
            all_associations += get_all_assoc(res[mod_keys[i]], mod_keys[i].split("_Assoc_dataset")[0])
            associations += get_assoc_consolidated(best_assoc_latent, best_assoc_model, lowestP_assoc_latent, lowestP_assoc_model, mod_keys[i].split("_Assoc_dataset")[0])
            i += 1
        
        #endswith "_LinRegrs_CV", "_LinRegrs_wFS_CV_FeatureSelection", "_Lasso_CV", "_Lasso_customL1_CV"
        else:
            lingregrs_cons = consolidate_metrics(res[mod_keys[i]])
            pheno = mod_keys[i].split("_LinRegrs")[0].split("_Lasso")[0]
            lingregrs_cons['Pheno'] = pheno
            lingregrs_cons['Model'] = mod_keys[i].split(pheno)[1][1:]
            model_results.append(lingregrs_cons)
            i += 1

    all_associations_df = pd.DataFrame(all_associations)
    associations_df = pd.DataFrame(associations)
    model_results_df = pd.concat(model_results)

    all_associations_df['ExpID'], all_associations_df['ExpParams'], all_associations_df['ResType']  = expID, exp_prams, resType
    associations_df['ExpID'], associations_df['ExpParams'], associations_df['ResType']  = expID, exp_prams, resType
    model_results_df['ExpID'], model_results_df['ExpParams'], model_results_df['ResType'] = expID, exp_prams, resType

    consolidated_model_results_df = model_results_df.groupby(['Pheno', 'Model', 'ExpID', 'ExpParams', 'ResType']).median(numeric_only=True).reset_index()

    return all_associations_df, associations_df, consolidated_model_results_df, model_results_df

In [10]:
all_associations = []
associations = []
consolidated_model_results = []
model_results = []

for exp in tqdm(exps):
    expID, resInfo = exp.replace(exp_root, "").split("/")[1:]
    exp_prams, resType = resInfo.split(expID)
    exp_prams = exp_prams[:-1]
    resType = resType[1:].replace("_results.pkl", "")
    
    maybe_complex_mode = resType.split("_")[0]
    if maybe_complex_mode in ["real", "imag", "mag", "phase", "cartesian", "polar", "dualcoords"]:
        expID += "_" + maybe_complex_mode
        resType = resType.replace(maybe_complex_mode + "_", "")

    with open(exp, "rb") as f:
        res = pickle.load(f)

    if len(res.keys()) > 0:
        all_associations_df, associations_df, consolidated_model_results_df, model_results_df = consolidate_res(res, expID, exp_prams, resType)
        all_associations.append(all_associations_df)
        associations.append(associations_df)
        consolidated_model_results.append(consolidated_model_results_df)
        model_results.append(model_results_df)

all_associations_df = pd.concat(all_associations)
associations_df = pd.concat(associations)
consolidated_model_results_df = pd.concat(consolidated_model_results)
model_results_df = pd.concat(model_results)

100%|██████████| 32/32 [05:07<00:00,  9.61s/it]


In [11]:
all_associations_df.to_csv(f"{out_root}/{exp_param}_all_assoc.csv", index=False)
associations_df.to_csv(f"{out_root}/{exp_param}_assoc.csv", index=False)
consolidated_model_results_df.to_csv(f"{out_root}/{exp_param}_consolidated_model_results.csv", index=False)
model_results_df.to_csv(f"{out_root}/{exp_param}_model_results.csv", index=False)

In [37]:
analysed_assoc_lowestP = compare_model_results_with_baseline(associations_df[(associations_df.Association=="LowestP") & (associations_df.ExpParams=="VIF_TScale_FScale")], mode="assoc")
analysed_assoc_best = compare_model_results_with_baseline(associations_df[(associations_df.Association=="Best") & (associations_df.ExpParams=="VIF_TScale_FScale")], mode="assoc")

In [39]:
model_results_df

,Pheno,Model,ExpID,ExpParams,ResType,MSE_TestSet,R-squared_TestSet,nFeat
0,10P_Liver_PDFF_(proton_density_fat_fraction),Lasso_CV,CAE_DualCoords_cartesian,TScale,Abdominal_composition_82779_MD_01_08_2024_16_5...,0.751329,0.212501,NaN
1,10P_Liver_PDFF_(proton_density_fat_fraction),Lasso_customL1_CV,CAE_DualCoords_cartesian,TScale,Abdominal_composition_82779_MD_01_08_2024_16_5...,0.776385,0.187074,NaN
2,10P_Liver_PDFF_(proton_density_fat_fraction),LinRegrs_CV,CAE_DualCoords_cartesian,TScale,Abdominal_composition_82779_MD_01_08_2024_16_5...,0.784983,0.178101,NaN
3,10P_Liver_PDFF_(proton_density_fat_fraction),RFELinRegrs_CV,CAE_DualCoords_cartesian,TScale,Abdominal_composition_82779_MD_01_08_2024_16_5...,0.782292,0.180067,206.0
4,Abdominal_fat_ratio,Lasso_CV,CAE_DualCoords_cartesian,TScale,Abdominal_composition_82779_MD_01_08_2024_16_5...,0.786555,0.221481,NaN
...,...,...,...,...,...,...,...,...
85,Visceral_fat_volume,Lasso_CV,BaseAE,VIF_TScale_FScale,Abdominal_organ_composition_82779_MD_01_08_202...,0.111822,0.888553,NaN
86,Visceral_fat_volume,Lasso_customL1_CV,BaseAE,VIF_TScale_FScale,Abdominal_organ_composition_82779_MD_01_08_202...,0.201887,0.799778,NaN
87,Visceral_fat_volume,LinRegrs_CV,BaseAE,VIF_TScale_FScale,Abdominal_organ_composition_82779_MD_01_08_202...,0.111668,0.888709,NaN
88,Visceral_fat_volume,LinRegrs_wFS_CV_FeatureSelection,BaseAE,VIF_TScale_FScale,Abdominal_organ_composition_82779_MD_01_08_202...,0.137640,0.860023,NaN


In [42]:
# analysed_model_results = compare_model_results_with_baseline(model_results_df[model_results_df.ExpParams==exp_param], mode="r2")
analysed_model_results = compare_model_results_with_baseline(model_results_df, mode="r2")

In [45]:
model_results_df

,Pheno,Model,ExpID,ExpParams,ResType,MSE_TestSet,R-squared_TestSet,nFeat
0,10P_Liver_PDFF_(proton_density_fat_fraction),Lasso_CV,CAE_DualCoords_cartesian,TScale,Abdominal_composition_82779_MD_01_08_2024_16_5...,0.751329,0.212501,NaN
1,10P_Liver_PDFF_(proton_density_fat_fraction),Lasso_customL1_CV,CAE_DualCoords_cartesian,TScale,Abdominal_composition_82779_MD_01_08_2024_16_5...,0.776385,0.187074,NaN
2,10P_Liver_PDFF_(proton_density_fat_fraction),LinRegrs_CV,CAE_DualCoords_cartesian,TScale,Abdominal_composition_82779_MD_01_08_2024_16_5...,0.784983,0.178101,NaN
3,10P_Liver_PDFF_(proton_density_fat_fraction),RFELinRegrs_CV,CAE_DualCoords_cartesian,TScale,Abdominal_composition_82779_MD_01_08_2024_16_5...,0.782292,0.180067,206.0
4,Abdominal_fat_ratio,Lasso_CV,CAE_DualCoords_cartesian,TScale,Abdominal_composition_82779_MD_01_08_2024_16_5...,0.786555,0.221481,NaN
...,...,...,...,...,...,...,...,...
85,Visceral_fat_volume,Lasso_CV,BaseAE,VIF_TScale_FScale,Abdominal_organ_composition_82779_MD_01_08_202...,0.111822,0.888553,NaN
86,Visceral_fat_volume,Lasso_customL1_CV,BaseAE,VIF_TScale_FScale,Abdominal_organ_composition_82779_MD_01_08_202...,0.201887,0.799778,NaN
87,Visceral_fat_volume,LinRegrs_CV,BaseAE,VIF_TScale_FScale,Abdominal_organ_composition_82779_MD_01_08_202...,0.111668,0.888709,NaN
88,Visceral_fat_volume,LinRegrs_wFS_CV_FeatureSelection,BaseAE,VIF_TScale_FScale,Abdominal_organ_composition_82779_MD_01_08_202...,0.137640,0.860023,NaN


In [44]:
analysed_model_results[1]

[('Spleen_iron_-_IDEAL', 'Lasso_customL1_CV'),
 ('Anterior_thigh_muscle_fat_infiltration_(MFI)_(left)', 'Lasso_customL1_CV'),
 ('Spleen_iron_-_protocol_normalised', 'Lasso_customL1_CV'),
 ('Liver_iron', 'Lasso_customL1_CV'),
 ('Spleen_iron_-_QC_indicator', 'Lasso_CV'),
 ('Spleen_iron_-_gradient_echo', 'Lasso_CV'),
 ('Liver_volume', 'Lasso_customL1_CV')]

In [ ]:
path_analyses_ml = "/group/glastonbury/soumick/MyCodes/GitLab/tricorder/partial_result_10P_Liver_PDFF_(proton_density_fat_fraction).pkl"
path_analyses_ml = "/project/ukbblatent/soumick/ML_analyses/F20259v3/v1.1.0_seventh_basket/CAE_Rupali/BaseAE/VIF_TScale_FScale_BaseAE_Abdominal_composition_82779_MD_01_08_2024_16_54_05_results.pkl"

with open(path_analyses_ml, 'rb') as f:
    ml_results = pickle.load(f)

In [ ]:
ml_results.keys()

dict_keys([])

In [ ]:
#define paths
path_analyses_ml = r"C:\MyFiles\UKBB_processed_latents\time2ch_4ChTrans_FactorVAE_latent64\MLAnalysis\cardiac_emb_results.pickle"
path_analyses_corr = r"C:\MyFiles\UKBB_processed_latents\time2ch_4ChTrans_FactorVAE_latent64\CorrAnalyses\cardiac_emb_correlations.csv"
path_analyses_corr_total = r"C:\MyFiles\UKBB_processed_latents\time2ch_4ChTrans_FactorVAE_latent64\CorrAnalyses\cardiac_emb_total_correlations.csv"

In [ ]:
#filters
r_squared_threshold = 0.5
f1_threshold = 0.3
f1_threshold_type = "macro avg" #Options: 'accuracy', 'macro avg', 'weighted avg', name of a specific class

In [ ]:
#load 
with open(path_analyses_ml, 'rb') as f:
    ml_results = pickle.load(f)

corr_results = pd.read_csv(path_analyses_corr)
corr_results_total = pd.read_csv(path_analyses_corr_total)

In [ ]:
corr_results[corr_results.isSignificant][corr_results.Strength == 'Strong']

In [ ]:
corr_results[corr_results.isSignificant][corr_results.Strength == 'Moderate']

In [ ]:
corr_results_total[corr_results_total.Strength == 'Strong']

In [ ]:
corr_results_total[corr_results_total.Strength == 'Strong']

In [ ]:
def get_keys(ml_results, exp):
    return {
        "exp": exp,
        "Tested": [k for k in ml_results.keys() if f"_{exp}_wTest" in k],
        "woTest": [k for k in ml_results.keys() if f"_{exp}_woTest" in k],
    }

In [ ]:
def sig_attrib_continuous(ml_results, keys, r_squared_threshold):
    sig_attrib = {}
    for i in range(len(keys['Tested'])):
        res = ml_results[keys['Tested'][i]]
        if res["R-squared_TestSet"] > r_squared_threshold:
            res_woTest = ml_results[keys['woTest'][i]]
            attribute = keys['woTest'][i].replace(f"_{keys['exp']}_woTest","")
            sig_attrib[attribute] = res_woTest['SignificantCoefficients']
    return sig_attrib

In [ ]:
def sig_attrib_categorical(ml_results, keys, f1_threshold, f1_threshold_type):
    sig_attrib = {}
    for i in range(len(keys['Tested'])):
        res = ml_results[keys['Tested'][i]]
        if res["ClassifRprt_TestSet"][res["ClassifRprt_TestSet"].index == f1_threshold_type]['f1-score'].item() > f1_threshold:
            res_woTest = ml_results[keys['woTest'][i]]
            attribute = keys['woTest'][i].replace(f"_{keys['exp']}_woTest","")
            for clcoeff in [k for k in res_woTest.keys() if "SignificantCoefficients_Cofficient_" in k]:
                sig_attrib[f"{attribute}_{clcoeff.replace('SignificantCoefficients_Cofficient_', '')}"] = res_woTest[clcoeff]
    return sig_attrib

In [ ]:
sig_attrib_lasso = sig_attrib_continuous(ml_results, get_keys(ml_results, "Lasso"), r_squared_threshold)
sig_attrib_singlelinrgrs = sig_attrib_continuous(ml_results, get_keys(ml_results, "LinRegrs"), r_squared_threshold)
sig_attrib_linrgrs_wFS = sig_attrib_continuous(ml_results, get_keys(ml_results, "LinRegrs_wFS"), r_squared_threshold)

In [ ]:
sig_attrib_lasso_cat = sig_attrib_categorical(ml_results, get_keys(ml_results, "LassoCat"), f1_threshold, f1_threshold_type)
sig_attrib_singlelogrgrs = sig_attrib_categorical(ml_results, get_keys(ml_results, "LogRegrs"), f1_threshold, f1_threshold_type)
sig_attrib_logrgrs_wFS = sig_attrib_categorical(ml_results, get_keys(ml_results, "LogRegrs_wFS"), f1_threshold, f1_threshold_type)

In [ ]:
sig_attrib